In [1]:
from static import *
from config import TransformerConfig
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from loss import frequency_loss, frequency_criterion
from data import split_data, train_val_split, anomaly_detection_data_provider
import os
from CATCH import CATCHModel
from torchinfo import summary
import pandas as pd
from earlyStopping import EarlyStopping
from torch.optim import lr_scheduler
import numpy as np
from utils import detect_validate, adjust_learning_rate, padding, validate_tfad, tfad_search_best_threshold
from predict import infer_score, infer_label
from evaluate import calculate
from copy import deepcopy
from tfad.model.outlier_exposure import coe_batch
from tfad.model.mixup import mixup_batch, slow_slope
from torch_optimizer import Yogi
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
dataset_name = "NYC"
mode = "score"
config = TransformerConfig(
    Mlr=1e-05,
    batch_size=128,
    cf_dim=64,
    d_ff=256,
    d_model=256,
    e_layers=3,
    head_dim=64,
    inference_patch_size=32,
    inference_patch_stride=1,
    lr=0.0001,
    n_heads=2,
    num_epochs=2,
    patch_size=16, 
    patch_stride=8,
    seq_len=192,
    temperature=0.07,
    # TFAD
    hp_lamb = 6400,
    # hyper-parameter for TCN encoder
    embedding_rep_dim = 64,
    tcn_kernel_size = 3,
    tcn_out_channels = 64,
    tcn_layers = 3,
    tcn_maxpool_out_channels = 2,
    normalize_embedding = True,
    suspect_window_length = 12,
    # hyper-parameter for classifier
    distance = "L2",
    # TFAD training hyper-parameters
    num_epochs_tfad = 1,
    lr_tfad = 1e-5,
    coe_rate = 0.5,
    mixup_rate = 0.3,
    slow_slop = 0,
    # TFAD validation hyper-parameters
    classifier_threshold = 0.5,
    val_labels_adj = True,
    threshold_grid_length_val = 0.1,
    # TFAD test hyper-parameters
    test_labels_adj = True,
    threshold_grid_length_test = 0.01,
)

scaler = StandardScaler()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.MSELoss()
auxi_loss_fn = frequency_loss(config)

# load data
train_data, train_label, test_data, test_label = split_data(
    os.path.join("data",dataset_name+".csv"),TRAIN_LENGTH[dataset_name]
)

# fit data
column_num = train_data.shape[1]
config.enc_in = column_num
config.dec_in = column_num
config.c_in = column_num
config.c_out = column_num
config.label_len = 48
model = CATCHModel(config)
# fix random seed again
from data import fix_random_seed
fix_random_seed(2021)
model.to(device)
# summary(model)
train_data_value, valid_data, train_label_value, valid_label = train_val_split(train_data, train_label, 0.8)
scaler.fit(train_data_value.values)

# data preprocess
train_data_value = pd.DataFrame(
    scaler.transform(train_data_value.values),
    columns=train_data_value.columns,
    index=train_data_value.index,
)
valid_data = pd.DataFrame(
    scaler.transform(valid_data.values),
    columns=valid_data.columns,
    index=valid_data.index,
)
valid_data_loader = anomaly_detection_data_provider(
    valid_data,
    valid_label,
    batch_size=config.batch_size,
    win_size=config.seq_len,
    step=1,
    mode="val",
)
train_data_loader = anomaly_detection_data_provider(
    train_data_value,
    train_label_value,
    batch_size=config.batch_size,
    win_size=config.seq_len,
    step=1,
    mode="train",
)

# train
early_stopping = EarlyStopping(patience=config.patience, verbose=True)
train_steps = len(train_data_loader)
main_params = [
    param for name, param in model.named_parameters() if 'mask_generator' not in name
]
optimizer = torch.optim.Adam(main_params, lr=config.lr)
optimizerM = torch.optim.Adam(model.mask_generator.parameters(), lr=config.Mlr)
scheduler = lr_scheduler.OneCycleLR(
    optimizer=optimizer,
    steps_per_epoch=train_steps,
    pct_start=config.pct_start,
    epochs=config.num_epochs,
    max_lr=config.lr,
)
schedulerM = lr_scheduler.OneCycleLR(
    optimizer=optimizerM,
    steps_per_epoch=train_steps,
    pct_start=config.pct_start,
    epochs=config.num_epochs,
    max_lr=config.Mlr,
)

In [3]:
print("---------------------CATCH TRAIN------------------------")
for epoch in range(config.num_epochs):
    iter_count = 0
    train_loss = []
    model.train()

    step = min(int(len(train_data_loader) / 10), 100)
    for i, (input, target) in enumerate(train_data_loader):
        iter_count += 1
        optimizer.zero_grad()
        input = input.float().to(device)
        outputs = model(input, mode="CATCH")
        output = outputs["z"][:, :, :]
        output_complex = outputs["complex_z"]
        dcloss = outputs["dcloss"]
        rec_loss = criterion(output, input)
        norm_input = model.revin_layer(input, 'transform')
        auxi_loss = auxi_loss_fn(output_complex, norm_input)
        loss = rec_loss + config.dc_lambda * dcloss + config.auxi_lambda * auxi_loss
        # print("RecLoss:{}, DCLoss:{}, AuxiLoss:{}".format(rec_loss, dcloss, auxi_loss))
        train_loss.append(loss.item())

        if (i + 1) % step == 0:
            optimizerM.step()
            optimizerM.zero_grad()

        if (i + 1) % 100 == 0:
            print(
                "\titers: {0}, epoch: {1} | training time loss: {2:.7f} | training fre loss: {3:.7f} | training dc loss: {4:.7f}".format(
                    i + 1,
                    epoch + 1,
                    rec_loss.item(),
                    auxi_loss.item(),
                    dcloss.item(),
                )
            )
            iter_count = 0

        loss.backward()
        optimizer.step()

    print("Epoch: {}".format(epoch + 1))
    train_loss = np.average(train_loss)
    valid_loss = detect_validate(model, valid_data_loader, criterion)
    print(
        "Epoch: {0}, Steps: {1} | Train Loss: {2:.7f} Vali Loss: {3:.7f}".format(
            epoch + 1, train_steps, train_loss, valid_loss
        )
    )

    early_stopping(valid_loss, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    adjust_learning_rate(optimizer, scheduler, epoch + 1, config)
    adjust_learning_rate(optimizerM, schedulerM, epoch + 1, config, printout=False)

# validate
model.load_state_dict(early_stopping.check_point)
model.to(device)
model.eval()
test = pd.DataFrame(
    scaler.transform(test_data.values), columns=test_data.columns, index=test_data.index
)
thre_loader = anomaly_detection_data_provider(
    test,
    None,
    batch_size=config.batch_size,
    win_size=config.seq_len,
    step=1,
    mode="thre",
)
test_data_loader = anomaly_detection_data_provider(
    test,
    None,
    batch_size=config.batch_size,
    win_size=config.seq_len,
    step=1,
    mode="test",
)
time_anomaly_criterion = nn.MSELoss(reduction='none')
freq_anomaly_criterion = frequency_criterion(config)
actual_label = test_label.to_numpy().flatten()

---------------------CATCH TRAIN------------------------


OutOfMemoryError: CUDA out of memory. Tried to allocate 530.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 11.98 GiB is allocated by PyTorch, and 21.19 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print("---------------------------CATCH TEST-------------------------")
if mode=="score":
    scores = infer_score(model, thre_loader, time_anomaly_criterion, freq_anomaly_criterion, config.score_lambda)
    predict_labels = deepcopy(scores)
    predict_labels, scores = padding(actual_label, predict_labels, scores)
    results = calculate(mode, actual_label.astype(float), predicted=predict_labels.astype(float))
    print(results)
elif mode=="label":
    if not isinstance(config.anomaly_ratio, list):
        config.anomaly_ratio = [config.anomaly_ratio]
    predict_labels, scores = infer_label(model, thre_loader, time_anomaly_criterion, freq_anomaly_criterion, config.score_lambda, train_data_loader, test_data_loader, config.anomaly_ratio)
    results_list = []
    for ratio, predict_label in predict_labels.items():
        predict_label, scores = padding(actual_label, predict_label, scores)
        results = calculate(mode, actual_label.astype(float), predicted=predict_label.astype(float))
        results["ratio"] = ratio
        results_list.append(results)
    print(results_list)

torch.save(model.state_dict(), dataset_name+".pth")
np.save(dataset_name+".npy", scores)
# 冻结CATCH模型参数
for name, param in model.named_parameters():
    if not name.startswith("TFAD"):
        param.requires_grad = False
# summary(model)

In [ ]:
print("----------------------------TFAD TRAIN-----------------------")
tfad_params = model.TFAD.parameters()
optimizer_tfad = Yogi(tfad_params, lr=config.lr_tfad)
tfad_criterion = nn.BCEWithLogitsLoss()
# 使用 mode="tfad-val"确保shuffle=False保证窗口顺序
tfad_val_loader = anomaly_detection_data_provider(
    valid_data,
    valid_label,
    batch_size=config.batch_size,
    win_size=config.seq_len,
    step=1,
    mode="tfad", 
)
tfad_test_loader = anomaly_detection_data_provider(
    test,
    test_label,
    batch_size=config.batch_size,
    win_size=config.seq_len,
    step=1,
    mode="tfad", 
)
best_f1 = 0.0
train_loss_list = []
f1_list = []
for epoch in range(config.num_epochs_tfad):
    train_loss_tfad = []
    model.train()
    progress_bar = tqdm(train_data_loader, desc=f"Epoch {epoch+1}/{config.num_epochs_tfad}")
    for i, (input, target) in enumerate(progress_bar):
        optimizer_tfad.zero_grad()
        # 1. 准备原始正常数据
        # input: [bs, seq_len, n_vars]
        x = input.float().to(device).permute(0, 2, 1) # [bs, n_vars, seq_len]
        
        # TFAD 官方逻辑：只根据 suspect_window_length (窗口末尾) 来判断标签
        # target: [bs, seq_len, 1]
        y = target[:, -config.suspect_window_length:, :].squeeze(-1).max(dim=1)[0].float().to(device)

        # 2. 数据增强 (生成伪异常)
        # COE
        if config.coe_rate > 0:
            x_oe, y_oe = coe_batch(
                x=x,
                y=y,
                coe_rate=config.coe_rate,
                suspect_window_length=config.suspect_window_length,
                random_start_end=True,
            )
            x = torch.cat((x, x_oe), dim=0)
            y = torch.cat((y, y_oe), dim=0)

        # Mixup
        if config.mixup_rate > 0.0:
            x_mixup, y_mixup = mixup_batch(
                x=x,
                y=y,
                mixup_rate=config.mixup_rate,
            )
            x = torch.cat((x, x_mixup), dim=0)
            y = torch.cat((y, y_mixup), dim=0)
            
        # Slow Slop
        if config.slow_slop > 0.0:
            x_slow, y_slow = slow_slope(
                x=x,
                y=y,
                mixup_rate=config.slow_slop,
            )
            x = torch.cat((x, x_slow), dim=0)
            y = torch.cat((y, y_slow), dim=0)
        
        # 3. 前向传播与 Loss 计算
        # 统一为[bs, seq_len, n_vars]形式输入
        logits_anomaly = model(x.permute(0, 2, 1), mode="TFAD")["TFAD_score"].squeeze()
        loss = tfad_criterion(logits_anomaly, y)
        
        loss.backward()
        optimizer_tfad.step()
        train_loss_tfad.append(loss.item())
        
        if (i + 1) % 100 == 0:
            print(f"Epoch: {epoch+1}, Iter: {i+1}, TFAD Loss: {np.mean(train_loss_tfad):.7f}")

    print(f"Train Loss: {np.mean(train_loss_tfad):.7f}")
    train_loss_list.append(np.mean(train_loss_tfad))
    # TFAD Validation
    tfad_labels = valid_label.to_numpy().astype(float).flatten()
    if sum(tfad_labels) == 0:
        if epoch == 0:
            print("Warning: No anomaly in validation set, use test set instead")
        tfad_probs = validate_tfad(model, tfad_test_loader, config, device)
        tfad_labels = test_label.to_numpy().astype(float).flatten()
    else:
        tfad_probs = validate_tfad(model, tfad_val_loader, config, device)
    metrics_best, threshold_best = tfad_search_best_threshold(tfad_probs, tfad_labels, config, stage="val")
    config.classifier_threshold = threshold_best
    print(f"Best threshold: {threshold_best:.4f}")
    print(f"Metrics: {metrics_best}")
    f1_list.append(metrics_best["f1"])
    if metrics_best["f1"] > best_f1:
        best_f1 = metrics_best["f1"]
        checkpoint = deepcopy(model.state_dict())

In [ ]:
print("----------------------------TFAD TEST-----------------------")
model.load_state_dict(checkpoint)
model.to(device)
model.eval()
tfad_labels = test_label.to_numpy().astype(float).flatten()
tfad_probs = validate_tfad(model, tfad_test_loader, config, device)
metrics_best, threshold_best = tfad_search_best_threshold(tfad_probs, tfad_labels, config, stage="test")
print(f"Best threshold: {threshold_best:.4f}")
print(f"Metrics: {metrics_best}")
print("---------------------------UNION TEST-----------------------")
scores_min = np.min(scores)
scores_max = np.max(scores)
scores_normalized = (scores - scores_min) / (scores_max - scores_min)
probs_min = np.nanmin(tfad_probs)
probs_max = np.nanmax(tfad_probs)
tfad_probs_normalized = (tfad_probs - probs_min) / (probs_max - probs_min)
tfad_probs_normalized = np.nan_to_num(tfad_probs_normalized, nan=0.0)
final_scores = np.maximum(scores_normalized, tfad_probs_normalized)
results = calculate(mode, actual_label.astype(float), predicted=final_scores.astype(float))
print(results)
metrics_best, threshold_best = tfad_search_best_threshold(final_scores, actual_label, config, stage="test")
print(f"Best threshold: {threshold_best:.4f}")
print(f"Metrics: {metrics_best}")
plt.subplot(2,1,1)
plt.plot(train_loss_list)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.subplot(2,1,2)
plt.plot(f1_list)
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("F1 Score Curve")
plt.show()